# Preprocesamiento: Monitoreo Lechería Paillaco
**Pilares aplicados:** Carga, Nulos, Duplicados, Normalización de texto, Formato de fechas, Outliers

## 1. Carga e Identificación del Dataset

In [1]:
import pandas as pd
import numpy as np

# Carga — 'NULL' literal se convierte a NaN automáticamente
df = pd.read_csv(
    r'C:\Users\ttvga\OneDrive\Escritorio\vel\Big Data\Clase 01\Side A\monitoreo_lecheria_paillaco.csv',
    na_values=['NULL']
)

print('=== ESTRUCTURA ORIGINAL ===')
print(f'Filas x Columnas: {df.shape}')
print(f'\nTipos de datos:\n{df.dtypes}')
print(f'\nPrimeras 5 filas:')
df.head()

=== ESTRUCTURA ORIGINAL ===
Filas x Columnas: (200, 5)

Tipos de datos:
ID_Vaca           object
Fecha_Control     object
Litros_Leche     float64
Temp_Corporal    float64
Estado_Salud      object
dtype: object

Primeras 5 filas:


,ID_Vaca,Fecha_Control,Litros_Leche,Temp_Corporal,Estado_Salud
0,VACA-138,03-11-2026,17.6,49.0,Tratamiento
1,VACA-112,2026-03-02,16.6,38.2,SALUDABLE
2,VACA-133,16/03/2026,15.4,48.1,En Observacion
3,VACA-129,15/03/2026,23.1,37.5,Tratamiento
4,VACA-137,03-24-2026,38.7,37.9,Tratamiento


In [2]:
print('=== ESTADÍSTICAS BÁSICAS ANTES DEL PREPROCESAMIENTO ===')
df.describe(include='all')

=== ESTADÍSTICAS BÁSICAS ANTES DEL PREPROCESAMIENTO ===


,ID_Vaca,Fecha_Control,Litros_Leche,Temp_Corporal,Estado_Salud
count,200,200,192.000000,200.000000,200
unique,18,68,NaN,NaN,7
top,VACA-138,Marzo 26,NaN,NaN,En Observacion
freq,31,44,NaN,NaN,35
mean,NaN,NaN,19.805729,39.373500,NaN
std,NaN,NaN,19.939913,2.807715,NaN
min,NaN,NaN,-39.100000,37.500000,NaN
25%,NaN,NaN,17.050000,38.100000,NaN
50%,NaN,NaN,24.500000,38.700000,NaN
75%,NaN,NaN,33.100000,39.200000,NaN


## 2. Valores Nulos

In [3]:
print('=== NULOS POR COLUMNA ===')
nulos = df.isnull().sum()
print(nulos)
print(f'\nTotal nulos: {nulos.sum()}')

print('\n--- Filas con nulos en Litros_Leche ---')
df[df['Litros_Leche'].isnull()]

=== NULOS POR COLUMNA ===
ID_Vaca          0
Fecha_Control    0
Litros_Leche     8
Temp_Corporal    0
Estado_Salud     0
dtype: int64

Total nulos: 8

--- Filas con nulos en Litros_Leche ---


,ID_Vaca,Fecha_Control,Litros_Leche,Temp_Corporal,Estado_Salud
18,VACA-111,Marzo 26,NaN,48.0,SALUDABLE
24,VACA-142,03-26-2026,NaN,39.1,saludable
66,VACA-137,2026-03-05,NaN,38.1,Tratamiento
103,VACA-119,16/03/2026,NaN,39.4,EN OBSERVACION
105,VACA-112,2026-03-16,NaN,38.8,En Observacion
116,VACA-147,2026-03-06,NaN,39.2,saludable
117,VACA-147,2026-03-06,NaN,39.2,saludable
163,VACA-150,25/03/2026,NaN,49.0,Saludable


In [4]:
# Imputación: mediana por vaca (más robusta que la media ante outliers)
df['Litros_Leche'] = df.groupby('ID_Vaca')['Litros_Leche'].transform(
    lambda x: x.fillna(x.median())
)

print(f'Nulos restantes en Litros_Leche: {df["Litros_Leche"].isnull().sum()}')

Nulos restantes en Litros_Leche: 0


## 3. Duplicados

In [5]:
n_antes = len(df)
print(f'Filas antes: {n_antes}')
print(f'Duplicados exactos encontrados: {df.duplicated().sum()}')

print('\n--- Muestra de filas duplicadas ---')
df[df.duplicated(keep=False)].sort_values(['ID_Vaca','Fecha_Control']).head(12)

Filas antes: 200
Duplicados exactos encontrados: 14

--- Muestra de filas duplicadas ---


,ID_Vaca,Fecha_Control,Litros_Leche,Temp_Corporal,Estado_Salud
155,VACA-100,25/03/2026,34.6,39.3,Tratamiento
156,VACA-100,25/03/2026,34.6,39.3,Tratamiento
61,VACA-120,26/03/2026,32.6,48.0,Tratamiento
62,VACA-120,26/03/2026,32.6,48.0,Tratamiento
124,VACA-120,28/03/2026,17.5,38.7,EN OBSERVACION
125,VACA-120,28/03/2026,17.5,38.7,EN OBSERVACION
177,VACA-121,02/03/2026,21.0,38.0,saludable
178,VACA-121,02/03/2026,21.0,38.0,saludable
179,VACA-121,02/03/2026,21.0,38.0,saludable
58,VACA-123,03-28-2026,36.3,39.5,En Observacion


In [6]:
df = df.drop_duplicates().reset_index(drop=True)

print(f'Filas después de eliminar duplicados: {len(df)}')
print(f'Filas eliminadas: {n_antes - len(df)}')

Filas después de eliminar duplicados: 186
Filas eliminadas: 14


## 4. Normalización de Texto — `Estado_Salud`

In [7]:
print('Valores únicos ANTES de normalizar:')
print(df['Estado_Salud'].value_counts())

Valores únicos ANTES de normalizar:
Estado_Salud
En Observacion    34
SALUDABLE         31
Tratamiento       26
Saludable         25
en observacion    24
saludable         24
EN OBSERVACION    22
Name: count, dtype: int64


In [8]:
# strip + title case → unifica todas las variantes
df['Estado_Salud'] = df['Estado_Salud'].str.strip().str.title()

print('Valores únicos DESPUÉS de normalizar:')
print(df['Estado_Salud'].value_counts())

Valores únicos DESPUÉS de normalizar:
Estado_Salud
Saludable         80
En Observacion    80
Tratamiento       26
Name: count, dtype: int64


## 5. Estandarización de Fechas — `Fecha_Control`

In [9]:
print('Formatos de fecha detectados (muestra):')
print(df['Fecha_Control'].value_counts().head(20))

Formatos de fecha detectados (muestra):
Fecha_Control
Marzo 26      41
2026-03-06     6
18/03/2026     5
03-11-2026     4
2026-03-21     4
2026-03-23     4
03-27-2026     4
14/03/2026     4
28/03/2026     4
2026-03-14     4
03-24-2026     4
25/03/2026     4
26/03/2026     4
03-06-2026     3
2026-03-18     3
2026-03-07     3
2026-03-02     3
03-19-2026     3
2026-03-05     3
17/03/2026     3
Name: count, dtype: int64


In [10]:
import re

# Mapa para meses en español → número
meses_es = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

def normalizar_fecha(fecha):
    """Convierte cualquier formato detectado a YYYY-MM-DD."""
    if pd.isnull(fecha):
        return pd.NaT
    
    fecha_str = str(fecha).strip()
    
    # Caso: 'Marzo 26' → asume año 2026
    match = re.match(r'^([A-Za-záéíóúÁÉÍÓÚ]+)\s+(\d{1,2})$', fecha_str)
    if match:
        mes_nombre = match.group(1).lower()
        dia = match.group(2).zfill(2)
        mes_num = meses_es.get(mes_nombre)
        if mes_num:
            return pd.to_datetime(f'2026-{mes_num}-{dia}', format='%Y-%m-%d')
        return pd.NaT
    
    # Caso: MM-DD-YYYY (formato americano, ej: 03-11-2026)
    match = re.match(r'^(\d{2})-(\d{2})-(\d{4})$', fecha_str)
    if match:
        return pd.to_datetime(fecha_str, format='%m-%d-%Y')
    
    # Caso: DD/MM/YYYY (ej: 16/03/2026)
    match = re.match(r'^(\d{1,2})/(\d{2})/(\d{4})$', fecha_str)
    if match:
        return pd.to_datetime(fecha_str, format='%d/%m/%Y')
    
    # Caso: YYYY-MM-DD (ISO estándar)
    try:
        return pd.to_datetime(fecha_str, format='%Y-%m-%d')
    except Exception:
        return pd.NaT

df['Fecha_Control'] = df['Fecha_Control'].apply(normalizar_fecha)

print('Tipo resultante:', df['Fecha_Control'].dtype)
print(f'Fechas nulas tras conversión: {df["Fecha_Control"].isnull().sum()}')
df['Fecha_Control'].describe()

Tipo resultante: datetime64[ns]
Fechas nulas tras conversión: 0


count                              186
mean     2026-03-17 11:13:32.903225856
min                2026-03-01 00:00:00
25%                2026-03-10 00:00:00
50%                2026-03-19 00:00:00
75%                2026-03-26 00:00:00
max                2026-03-28 00:00:00
Name: Fecha_Control, dtype: object

## 6. Outliers

### 6a. `Litros_Leche` — Valores negativos (imposibles físicamente)

In [11]:
negativos_leche = df[df['Litros_Leche'] < 0]
print(f'Registros con Litros_Leche negativo: {len(negativos_leche)}')
negativos_leche[['ID_Vaca','Fecha_Control','Litros_Leche']]

Registros con Litros_Leche negativo: 27


,ID_Vaca,Fecha_Control,Litros_Leche
5,VACA-111,2026-03-13,-19.90
6,VACA-142,2026-03-26,-38.20
13,VACA-119,2026-03-26,-32.40
20,VACA-132,2026-03-26,-18.70
24,VACA-142,2026-03-26,-0.05
29,VACA-149,2026-03-26,-35.90
52,VACA-139,2026-03-26,-26.50
56,VACA-142,2026-03-23,-21.40
75,VACA-112,2026-03-01,-21.50
89,VACA-119,2026-03-26,-37.80


In [12]:
# Corrección: error de signo → valor absoluto
df['Litros_Leche'] = df['Litros_Leche'].abs()

print(f'Negativos restantes: {(df["Litros_Leche"] < 0).sum()}')
print(f'Rango final: [{df["Litros_Leche"].min()}, {df["Litros_Leche"].max()}]')

Negativos restantes: 0
Rango final: [0.049999999999998934, 39.8]


### 6b. `Temp_Corporal` — Valores sobre 45°C (error tipográfico probable: 38.x ingresado como 48.x)

In [13]:
outliers_temp = df[df['Temp_Corporal'] > 42]
print(f'Registros con Temp_Corporal > 42°C: {len(outliers_temp)}')
outliers_temp[['ID_Vaca','Fecha_Control','Temp_Corporal','Estado_Salud']]

Registros con Temp_Corporal > 42°C: 15


,ID_Vaca,Fecha_Control,Temp_Corporal,Estado_Salud
0,VACA-138,2026-03-11,49.0,Tratamiento
2,VACA-133,2026-03-16,48.1,En Observacion
8,VACA-110,2026-03-14,48.0,En Observacion
15,VACA-133,2026-03-26,48.7,En Observacion
18,VACA-111,2026-03-26,48.0,Saludable
22,VACA-112,2026-03-11,47.8,En Observacion
38,VACA-138,2026-03-27,47.7,Tratamiento
57,VACA-120,2026-03-26,48.0,Tratamiento
67,VACA-100,2026-03-23,48.2,Saludable
70,VACA-150,2026-03-26,48.1,En Observacion


In [14]:
# Corrección: restar 10 (47.x → 37.x, 48.x → 38.x, 49.x → 39.x)
df.loc[df['Temp_Corporal'] > 42, 'Temp_Corporal'] = \
    df.loc[df['Temp_Corporal'] > 42, 'Temp_Corporal'] - 10

print(f'Outliers restantes (>42°C): {(df["Temp_Corporal"] > 42).sum()}')
print(f'Rango final Temp_Corporal: [{df["Temp_Corporal"].min()}, {df["Temp_Corporal"].max()}]')

Outliers restantes (>42°C): 0
Rango final Temp_Corporal: [37.5, 39.5]


## 7. Verificación Final

In [15]:
print('=== DATASET FINAL ===')
print(f'Shape: {df.shape}')
print(f'\nNulos por columna:\n{df.isnull().sum()}')
print(f'\nDuplicados: {df.duplicated().sum()}')
print(f'\nEstados únicos:\n{df["Estado_Salud"].value_counts()}')
print(f'\nRango fechas: {df["Fecha_Control"].min()} → {df["Fecha_Control"].max()}')
print(f'\nRango Litros_Leche: {df["Litros_Leche"].min()} – {df["Litros_Leche"].max()}')
print(f'Rango Temp_Corporal: {df["Temp_Corporal"].min()} – {df["Temp_Corporal"].max()}')

df.describe()

=== DATASET FINAL ===
Shape: (186, 5)

Nulos por columna:
ID_Vaca          0
Fecha_Control    0
Litros_Leche     0
Temp_Corporal    0
Estado_Salud     0
dtype: int64

Duplicados: 0

Estados únicos:
Estado_Salud
Saludable         80
En Observacion    80
Tratamiento       26
Name: count, dtype: int64

Rango fechas: 2026-03-01 00:00:00 → 2026-03-28 00:00:00

Rango Litros_Leche: 0.049999999999998934 – 39.8
Rango Temp_Corporal: 37.5 – 39.5


,Fecha_Control,Litros_Leche,Temp_Corporal
count,186,186.000000,186.000000
mean,2026-03-17 11:13:32.903225856,26.778226,38.514516
min,2026-03-01 00:00:00,0.050000,37.500000
25%,2026-03-10 00:00:00,19.950000,38.000000
50%,2026-03-19 00:00:00,27.050000,38.600000
75%,2026-03-26 00:00:00,34.075000,39.000000
max,2026-03-28 00:00:00,39.800000,39.500000
std,NaN,7.904273,0.580078


In [16]:
print('Primeras 10 filas del dataset limpio:')
df.head(10)

Primeras 10 filas del dataset limpio:


,ID_Vaca,Fecha_Control,Litros_Leche,Temp_Corporal,Estado_Salud
0,VACA-138,2026-03-11,17.6,39.0,Tratamiento
1,VACA-112,2026-03-02,16.6,38.2,Saludable
2,VACA-133,2026-03-16,15.4,38.1,En Observacion
3,VACA-129,2026-03-15,23.1,37.5,Tratamiento
4,VACA-137,2026-03-24,38.7,37.9,Tratamiento
5,VACA-111,2026-03-13,19.9,39.4,Tratamiento
6,VACA-142,2026-03-26,38.2,37.6,En Observacion
7,VACA-110,2026-03-21,22.0,38.2,Saludable
8,VACA-110,2026-03-14,35.7,38.0,En Observacion
9,VACA-111,2026-03-28,27.0,37.8,Tratamiento


## 8. Exportar Dataset Limpio

In [17]:
ruta_salida = r'C:\Users\ttvga\OneDrive\Escritorio\vel\Big Data\Clase 01\Side A\monitoreo_lecheria_paillaco_LIMPIO.csv'

df.to_csv(ruta_salida, index=False, date_format='%Y-%m-%d')
print(f'Dataset limpio exportado → {ruta_salida}')

Dataset limpio exportado → C:\Users\ttvga\OneDrive\Escritorio\vel\Big Data\Clase 01\Side A\monitoreo_lecheria_paillaco_LIMPIO.csv
